# Projection and Predicate pushdown in Apache Parquet using Spark


In [ ]:
%run_nb spark-start

In [ ]:
path="$HOME/work/data/kaggle/datasets/stefanoleone992/fifa-23-complete-player-dataset"
!ls -la {path}

In [ ]:
csv_file = Path(path) / "male_players.csv"
parquet_output = Path(path) / "parquet" / "male_players"

In [ ]:
%load_ext autotime

In [ ]:
if not parquet_output.is_dir():
    print(f"Convert CSV to parquet")
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(str(csv_file))
    )

    df.write.mode("overwrite").parquet(str(parquet_output))
 
else:
    print(f"Read parquet")
    
    df = (
        spark.read
        .parquet(str(parquet_output))
    )      

In [ ]:
!echo "Number of files: $(find {path}/parquet/male_players/*.parquet | wc -l)"
!echo "Files: $(ls {path}/parquet/male_players/*.parquet)"

In [ ]:
humanize.intword(df.count())

In [ ]:
df.explain()

# Predicate pushdown

In [ ]:
df_all = spark.read.parquet(str(parquet_output))

df_select = (
    spark.read
    .parquet(str(parquet_output))
    .select("value_eur")
)

In [ ]:
df_select_filter = (
    spark.read
    .parquet(str(parquet_output))
    .filter(F.col("value_eur") > 1000000)
    .select("value_eur")
)

In [ ]:
humanize.intword(df_select_filter.count())

In [ ]:
humanize.intword(df_select.count())

In [ ]:
viewdf(df_select_filter, limit=5)

# Physical plan
## See "PushedFilters:"

In [ ]:
df_select_filter.explain("formatted")

In [ ]:
df_select.explain("formatted")

In [ ]:
humanize.intword(df_select_filter.count())

In [ ]:
humanize.intword(df_select.count())

In [ ]:
def benchmark(label, query):
    # Run once to reduce JVM / planning noise
    query.collect()

    start = time.perf_counter()
    result = query.collect()
    elapsed = time.perf_counter() - start

    print(f"{label}: {elapsed:.3f} s")
    #return result

In [ ]:
benchmark("Only value_eur", df_select)

In [ ]:
df_select_filter.explain("simple")

In [ ]:
df_select_filter.explain("formatted")

In [ ]:
df_select_filter.explain("extended")

In [ ]:
df_select_filter.explain("codegen")

In [ ]:
df_select_filter.explain("cost")

In [ ]:
benchmark("Filtered, with predicate pushdown", df_select_filter)

In [ ]:
%run_nb spark-show